# Source Preparation


---

Creating Heterogenious source data for Data ingestion in Azure Data Lake Storage.

We have Olist E-commerce CSV files on github.
Github Link : [Saqib Souran](https://github.com/saqib-souran/Brazilian-Ecommerce-Azure-Pipeline/tree/c891935ee0d1df009ea70f3a689298432c6d8e3b/data)

Project: Azure E-commerce data Pipeline

Done by: Saqib Souran

---


## MySQL Source Preparation
Send olist_order_payments_dataset.csv to online databse hosted at filess.io.


**Flow** :

olist_order_payments_dataset.csv --> MySQL --> [Azure ADLS via Azure Data Factory]




In [1]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 26.5 MB/s eta 0:00:00


In [18]:
import pandas as pd
url = "https://raw.githubusercontent.com/saqib-souran/Brazilian-Ecommerce-Azure-Pipeline/refs/heads/main/data/olist_order_payments_dataset.csv"
olist = pd.read_csv(url,index_col=1)
olist.head()


,order_id,payment_type,payment_installments,payment_value
payment_sequential,,,,
1,b81ef226f3fe1789b1e8b2acac839d17,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,credit_card,1,24.39
1,25e8ea4e93396b6fa0d3dd708e76c1bd,credit_card,1,65.71
1,ba78997921bbcdc1373bb41e913ab953,credit_card,8,107.78
1,42fdf880ba16b47b59251dd489d4441a,credit_card,2,128.45


In [19]:
olist.shape

(103886, 4)

I changed Database name and password to random

In [ ]:
import mysql.connector
from mysql.connector import Error

hostname = "pj8b79.h.filess.io"
database = "oliseEWD"
port = "3306"
username = "Souran_random_db"
password = "myrandompassword"

#CSV file path
csv_file_path = 'olist_order_payments_dataset.csv'

#Table name wher we will upload our data
table_name = 'olist_order_payments'


try:
    # connection to mySQL server
    connection = mysql.connector.connect(host=hostname, database=database, user=username, password=password, port=port)
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        #cursor to execute SQL queries
        cursor = connection.cursor()

        #for Clean insertion Dropping table (if already exists)
        cursor.execute(f"DROP TABLE IF EXISTS {table_name};")
        print(f"Table '{table_name}' dropped if it existed.")

        #Create table structure to match csv file
        create_table_query = f"""
        CREATE TABLE {table_name} (
            order_id VARCHAR(50),
            payment_sequential INT,
            payment_type VARCHAR(20),
            payment_installments INT,
            payment_value FLOAT
        );
        """
        cursor.execute(create_table_query)
        print(f"Table '{table_name}' created successfully.")

        #Load data to pd dataframe
        data = pd.read_csv(csv_file_path)

        #Insert data in batches of 1000 records
        batch_size = 1000
        total_records = len(data)

        for i in range(0, total_records, batch_size):
          batch = data.iloc[i:i+batch_size]

          # Converting into tuples fro MySQL insertion
          batch_records = [
              tuple(row) for row in batch.itertuples(index=False, name=None)
          ]

          # insert query
          insert_query = f"""INSERT INTO {table_name}
          (order_id, payment_sequential, payment_type, payment_installments, payment_value)
          VALUES (%s, %s, %s, %s, %s);"""

          #Execute insert query for the batch
          cursor.executemany(insert_query, batch_records)
          connection.commit() # commit after each batch
          print(f"Inserted records from {i+1} to {min((i+batch_size, total_records))} into '{table_name}' successfully.")

        print(f"All {total_records} records insert successfully into '{table_name}'")

except Error as e:
    print("Error while connecting to MySQL", e)
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL connection is closed")


/tmp/ipykernel_4100/1256913172.py:21: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()


Connected to MySQL Server version  5.7.44
Table 'olist_order_payments' dropped if it existed.
Table 'olist_order_payments' created successfully.
Inserted records from 1 to 1000 into 'olist_order_payments' successfully.
Inserted records from 1001 to 2000 into 'olist_order_payments' successfully.
Inserted records from 2001 to 3000 into 'olist_order_payments' successfully.
Inserted records from 3001 to 4000 into 'olist_order_payments' successfully.
Inserted records from 4001 to 5000 into 'olist_order_payments' successfully.
Inserted records from 5001 to 6000 into 'olist_order_payments' successfully.
Inserted records from 6001 to 7000 into 'olist_order_payments' successfully.
Inserted records from 7001 to 8000 into 'olist_order_payments' successfully.
Inserted records from 8001 to 9000 into 'olist_order_payments' successfully.
Inserted records from 9001 to 10000 into 'olist_order_payments' successfully.
Inserted records from 10001 to 11000 into 'olist_order_payments' successfully.
Inserted

## MongoDB Source Preparation

Send product_category_name_translation.csv to online databse hosted at filess.io.


**Flow** :

product_category_name_translation.csv --> MongoDB --> [Azure ADLS via Azure Data Factory]

In [5]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 25.0 MB/s eta 0:00:00


I changed username and password to random

In [ ]:
# importing module
from pymongo import MongoClient

hostname = "6znl6a.h.filess.io"
database = "olistDB_Nosql_sheeasily"
port = "27018"
username = "UUUUUUUUUUUUUUUUU"
password = "HHHHHHHHHHHHHHHHHHH"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]
print(mydatabase.name)

client.close()


olistDB_Nosql_sheeasily


In [8]:
import pandas as pd

# Path to the CSV file
url = 'https://raw.githubusercontent.com/saqib-souran/Brazilian-Ecommerce-Azure-Pipeline/refs/heads/main/data/product_category_name_translation.csv'
collection_name = 'product_category_name_translation'

try:
    # Connect with the portnumber and host
    client = MongoClient(uri)

    # Access database
    mydatabase = client[database]
    print(f"Connected to Database: {mydatabase.name}")
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(url, index_col=1)
    print(f"Loaded {len(df)} records from {url}")

    # Convert DataFrame to list of dictionaries for MongoDB insertion
    records = df.to_dict(orient='records')

    # Select the collection
    collection = mydatabase[collection_name]

    # Drop collection if it already exists to avoid duplication
    collection.drop()
    print(f"Collection '{collection_name}' dropped for a clean insertion.")

    # Insert records into MongoDB
    result = collection.insert_many(records)
    print(f"Successfully ingested {len(result.inserted_ids)} records into MongoDB collection '{collection_name}'.")

except Exception as e:
    print("An error occurred during ingestion:", e)

finally:
  if client:
    client.close()
    print("Connection Closed")

Connected to Database: olistDB_Nosql_sheeasily
Loaded 71 records from product_category_name_translation.csv
Collection 'product_category_name_translation' dropped for a clean insertion.
Successfully ingested 71 records into MongoDB collection 'product_category_name_translation'.
Connection Closed
